In [1]:
!pip install librosa


In [2]:
import numpy as np

fps = 3
duration_seconds = 90 
total_frames = fps * duration_seconds
cv_features = 512

dummy_cv_array = np.random.rand(total_frames, cv_features).astype(np.float32)

file_name = 'dummy_cv_output.npy'
np.save(file_name, dummy_cv_array)

print("CV OUTPUT SIMULATION")
print(f"File created: {file_name}")
print(f"Exact Shape of the data: {dummy_cv_array.shape}")
print(f"First 3 numbers of the very first frame: {dummy_cv_array[0, :3]}")

CV OUTPUT SIMULATION
File created: dummy_cv_output.npy
Exact Shape of the data: (270, 512)
First 3 numbers of the very first frame: [0.1959024  0.52325815 0.5890123 ]


In [3]:
import numpy as np
import librosa
duration_seconds = 90
fps = 3
target_frames = duration_seconds * fps  
n_mfcc = 128                            

sr = 22050  
hop_length = int(sr / fps)

raw_audio = np.random.randn(sr * duration_seconds).astype(np.float32)

audio_features = librosa.feature.mfcc(
    y=raw_audio, 
    sr=sr, 
    n_mfcc=n_mfcc, 
    hop_length=hop_length
).T

audio_features = audio_features[:target_frames, :]

file_name = 'dummy_audio_output.npy'
np.save(file_name, audio_features)

print("--- AUDIO EXTRACTION SIMULATION ---")
print(f"File created: {file_name}")
print(f"Exact Shape of the data: {audio_features.shape}")
print(f"First 3 MFCCs of the very first frame: {audio_features[0, :3]}")

--- AUDIO EXTRACTION SIMULATION ---
File created: dummy_audio_output.npy
Exact Shape of the data: (270, 128)
First 3 MFCCs of the very first frame: [163.32794  -13.73204  -11.212271]


In [5]:
import torch
import torch.nn as nn

class LatentTrajectoryLSTM(nn.Module):
    def __init__(self, cv_dim=512, audio_dim=128, hidden_dim=256):
        super(LatentTrajectoryLSTM, self).__init__()
        
        # 1. The Dimensionality Aligners (Projecting both to 256)
        self.cv_projection = nn.Linear(cv_dim, hidden_dim)
        self.audio_projection = nn.Linear(audio_dim, hidden_dim)
        
        # 2. The Sequence Engine (Takes fused 512 back down to 256 over time)
        # batch_first=True means it expects tensors in shape (Batch, Time, Features)
        self.lstm = nn.LSTM(input_size=hidden_dim * 2, hidden_size=hidden_dim, batch_first=True)
        
        # 3. The Final Scorer (Translates the 3D embedding into a 0.0 - 1.0 Anomaly Score)
        self.score_layer = nn.Linear(3, 1)
        
    def forward(self, cv_x, audio_x):
        # cv_x shape: (Batch, Time, 512)
        # audio_x shape: (Batch, Time, 128)
        
        # --- STEP A: Latent Projection ---
        h_video = self.cv_projection(cv_x)     # Shape: (Batch, Time, 256)
        h_audio = self.audio_projection(audio_x) # Shape: (Batch, Time, 256)
        
        # --- STEP B: Calculate Audio-Video Sync Residuals ---
        # Mathematical distance between video and audio in the shared latent space
        # L2 Norm (Euclidean distance) across the feature dimension
        sync_residuals = torch.norm(h_video - h_audio, p=2, dim=-1) # Shape: (Batch, Time)
        mean_sync_residual = sync_residuals.mean(dim=1)             # Shape: (Batch,)
        
        # --- STEP C: The Temporal Fusion ---
        # Stitch them together to create the full multimodal state
        fused_state = torch.cat([h_video, h_audio], dim=-1)         # Shape: (Batch, Time, 512)
        
        # Pass the fused state through the LSTM to get the timeline trajectory
        trajectory, _ = self.lstm(fused_state)                      # Shape: (Batch, Time, 256)
        
        # --- STEP D: Calculate Latent Velocity ---
        # 1st Derivative: Distance between Frame T and Frame T-1
        velocity = trajectory[:, 1:, :] - trajectory[:, :-1, :]     # Shape: (Batch, Time-1, 256)
        velocity_mag = torch.norm(velocity, p=2, dim=-1)            # Magnitude
        velocity_variance = velocity_mag.var(dim=1)                 # Variance of speed (Batch,)
        
        # --- STEP E: Calculate Acceleration Spikes ---
        # 2nd Derivative: Change in velocity between steps
        acceleration = velocity[:, 1:, :] - velocity[:, :-1, :]     # Shape: (Batch, Time-2, 256)
        acceleration_mag = torch.norm(acceleration, p=2, dim=-1)
        max_acceleration = acceleration_mag.max(dim=1)[0]           # Peak jitter (Batch,)
        
        # --- STEP F: The Final Output Contracts ---
        # Build the exact 3-dimensional array requested by the Classifier teammate
        temporal_embedding = torch.stack([
            velocity_variance, 
            max_acceleration, 
            mean_sync_residual
        ], dim=1)                                                   # Shape: (Batch, 3)
        
        # Push the 3 physics metrics through a final sigmoid to get a 0-1 probability
        raw_score = self.score_layer(temporal_embedding)
        anomaly_score = torch.sigmoid(raw_score).squeeze()          # Shape: Scalar float
        
        return anomaly_score, temporal_embedding

# --- QUICK VERIFICATION TEST ---
print("--- PYTORCH LSTM INSTANTIATION ---")
# Create the model
model = LatentTrajectoryLSTM()

# Load our dummy data from the previous Kaggle cells and convert to PyTorch Tensors
# We add unsqueeze(0) to simulate a Batch Size of 1 (a single video)
dummy_cv_tensor = torch.tensor(np.load('dummy_cv_output.npy')).unsqueeze(0) 
dummy_audio_tensor = torch.tensor(np.load('dummy_audio_output.npy')).unsqueeze(0)

# Run the forward pass!
score, embedding = model(dummy_cv_tensor, dummy_audio_tensor)

print(f"Temporal Anomaly Score Output: {score.item():.4f} (Float between 0 and 1)")
print(f"Temporal Embedding Output: {embedding.detach().numpy().flatten()} (3D Array)")

--- PYTORCH LSTM INSTANTIATION ---
Temporal Anomaly Score Output: 1.0000 (Float between 0 and 1)
Temporal Embedding Output: [7.4427173e-02 4.2948523e+00 1.6546498e+02] (3D Array)
